# Classification des Cultures Agricoles par Indices de Végétation Avancés

## Introduction
La sécurité alimentaire en RDC dépend d'un suivi précis des zones de cultures. Ce notebook utilise des indices de végétation robustes (NDVI et EVI) pour différencier les types de couverts végétaux et identifier les parcelles agricoles même dans des conditions atmosphériques humides.

## Objectifs
*   **Zonage agricole** : Isoler les parcelles cultivées de la végétation naturelle.
*   **Santé des cultures** : Utiliser l'EVI (Enhanced Vegetation Index) pour une mesure plus fine de la biomasse.
*   **Estimation de surface** : Calculer les superficies nettes cultivées.

## Méthodologie
1.  **Setup** : Installation des outils géospatiaux.
2.  **Acquisition** : Images Sentinel-2 (Bandes Bleu, Rouge, NIR).
3.  **Calcul multi-indices** : Application des formules NDVI et EVI.
4.  **Classification** : Seuil d'identification des parcelles agricoles actives.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration
# ====================================================
!pip install geemap earthengine-api rasterio matplotlib seaborn -q

import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ Système prêt')

## Zone d'Étude (ROI)
Focus sur une zone agricole régionale.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition des Données
Nous exportons les bandes nécessaires pour le NDVI et l'EVI.

In [ ]:
# ====================================================
# ÉTAPE 3 : Données Sentinel-2
# ====================================================
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(roi).median().clip(roi))

geemap.ee_export_image(image.select(['B2', 'B4', 'B8']), 'crops.tif', scale=30, region=roi)

## Calcul des Indices Agricoles
Le NDVI mesure la quantité de végétation, tandis que l'EVI (Enhanced Vegetation Index) offre une meilleure sensibilité dans les zones à haute densité de biomasse en réduisant les influences atmosphériques.

In [ ]:
# ====================================================
# ÉTAPE 4 : Calcul NDVI et EVI
# ====================================================
with rasterio.open('crops.tif') as src: data = src.read().astype(np.float32)
blue, red, nir = data[0], data[1], data[2]
ndvi = (nir - red) / (nir + red + 1e-8)
evi = 2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))

agricultural_pixels = (ndvi > 0.5) * evi

plt.figure(figsize=(10, 8))
plt.imshow(agricultural_pixels, cmap='YlGn')
plt.colorbar(label='Indice d'intensité de culture (EVI)')
plt.title('Classification des Zones de Cultures Agricoles')
plt.axis('off')
plt.show()